# Bay Area Gentrification Risk — Choropleth Map

Produces an interactive folium map of all census tracts in San Francisco, Oakland (Alameda County), and San Jose (Santa Clara County), color-coded by relative gentrification risk.

- **White** = lowest relative risk (scaled 0)
- **Red** = highest relative risk (scaled 1)
- Scores are min-max scaled across all tracts so the ranking is meaningful, not the absolute value.
- Tracts outside the model (rest of Alameda / Santa Clara counties) are shown in light gray.

## 1 — Install packages

In [ ]:
!pip install -q geopandas folium branca mapclassify

## 2 — Imports

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
import branca.colormap as cm
import json
import requests
import zipfile
import io
from pathlib import Path

print('All packages imported.')

## 3 — Mount Drive & Load Risk Scores

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Adjust this path to wherever your CSVs live ───────────────────────────────
HERE = Path('/content/drive/MyDrive/1. MS&E 125/MS&E 125 Project/Full Model Comparison')

risk_df = pd.read_csv(HERE / 'gentrification_risk_scores_v2.csv')
risk_df['tract_id'] = risk_df['tract_id'].astype(str).str.zfill(11)

print(f'Risk scores loaded: {len(risk_df)} tracts')
print(risk_df[['tract_id', 'city', 'risk_score', 'top_risk_driver']].head())

## 4 — Download Census Tract Boundaries

Downloads the 2020 California cartographic boundary file (~7 MB) directly from the Census Bureau. This only needs to run once per Colab session.

In [ ]:
TRACT_URL = 'https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_06_tract_500k.zip'
EXTRACT_DIR = '/content/ca_tracts'

print('Downloading California tract boundaries (~7 MB)...')
r = requests.get(TRACT_URL, timeout=60)
r.raise_for_status()

with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    z.extractall(EXTRACT_DIR)

gdf_ca = gpd.read_file(f'{EXTRACT_DIR}/cb_2020_06_tract_500k.shp')
gdf_ca['tract_id'] = gdf_ca['GEOID'].astype(str).str.zfill(11)

print(f'California tracts loaded: {len(gdf_ca)} total')
print('CRS:', gdf_ca.crs)

## 5 — Filter to Bay Area Counties & Merge Risk Scores

In [ ]:
# Counties covered by the model
# 001 = Alameda (Oakland), 075 = San Francisco, 085 = Santa Clara (San Jose)
BAY_COUNTIES = {'001', '075', '085'}

bay_gdf = gdf_ca[gdf_ca['COUNTYFP'].isin(BAY_COUNTIES)].copy()
bay_gdf = bay_gdf.to_crs('EPSG:4326')   # WGS84 for folium

print(f'Bay Area tracts: {len(bay_gdf)}')

# Merge risk scores — tracts not in the model get NaN
merged = bay_gdf.merge(
    risk_df[['tract_id', 'risk_score', 'city', 'top_risk_driver', 'driver_direction',
             'lr_risk_score', 'rf_risk_score', 'xgb_risk_score']],
    on='tract_id',
    how='left'
)

# Min-max scale so highest-risk tract = 1, lowest = 0
scored = merged['risk_score'].dropna()
r_min, r_max = scored.min(), scored.max()

merged['scaled_risk'] = (merged['risk_score'] - r_min) / (r_max - r_min)
merged['in_model']    = merged['risk_score'].notna()

# Fill display columns for tooltip
merged['city']            = merged['city'].fillna('Outside model')
merged['top_risk_driver'] = merged['top_risk_driver'].fillna('—')
merged['driver_direction']= merged['driver_direction'].fillna('—')
merged['scaled_risk']     = merged['scaled_risk'].fillna(-1)   # sentinel for gray
merged['risk_score_disp'] = merged['risk_score'].apply(
    lambda x: f'{x:.4f}' if pd.notna(x) else 'N/A'
)
merged['scaled_risk_disp'] = merged['scaled_risk'].apply(
    lambda x: f'{x:.4f}' if x >= 0 else 'N/A'
)

in_model_count = merged['in_model'].sum()
print(f'Tracts with risk scores : {in_model_count}')
print(f'Tracts outside model    : {(~merged["in_model"]).sum()}')
print(f'\nRisk score range (raw)  : {r_min:.4f} – {r_max:.4f}')
print(f'Scaled range            : 0.0000 – 1.0000')

## 6 — Build the Choropleth Map

In [ ]:
# ── Colormap: white → red ──────────────────────────────────────────────────────
colormap = cm.LinearColormap(
    colors=['#ffffff', '#ff4444', '#cc0000'],
    vmin=0, vmax=1,
    caption='Relative Gentrification Risk  (0 = lowest, 1 = highest)'
)

def tract_color(scaled):
    """Return fill color. Gray for tracts outside the model."""
    if scaled < 0:
        return '#cccccc'
    return colormap(scaled)


# ── Style function ─────────────────────────────────────────────────────────────
def style_fn(feature):
    scaled = feature['properties'].get('scaled_risk', -1)
    in_model = feature['properties'].get('in_model', False)
    return {
        'fillColor':   tract_color(scaled),
        'color':       '#888888' if not in_model else '#555555',
        'weight':      0.4,
        'fillOpacity': 0.4 if not in_model else 0.85,
    }

def highlight_fn(feature):
    return {
        'color':       '#000000',
        'weight':      2,
        'fillOpacity': 0.95,
    }


# ── Build GeoJSON with properties ─────────────────────────────────────────────
# Select only the columns folium needs (keeps GeoJSON small)
export_cols = [
    'geometry', 'tract_id', 'city', 'scaled_risk', 'scaled_risk_disp',
    'risk_score_disp', 'in_model', 'top_risk_driver', 'driver_direction'
]
geo_export = merged[export_cols].copy()
geojson_str = geo_export.to_json()


# ── Folium map ─────────────────────────────────────────────────────────────────
m = folium.Map(
    location=[37.55, -122.05],
    zoom_start=10,
    tiles='CartoDB positron'
)

folium.GeoJson(
    geojson_str,
    name='Gentrification Risk',
    style_function=style_fn,
    highlight_function=highlight_fn,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            'tract_id', 'city',
            'scaled_risk_disp', 'risk_score_disp',
            'top_risk_driver', 'driver_direction'
        ],
        aliases=[
            'Tract ID:', 'City:',
            'Relative Risk (0–1):', 'Raw Ensemble Score:',
            'Top Risk Driver:', 'Direction:'
        ],
        sticky=True,
        style=(
            'background-color: white; color: #333; '
            'font-family: Arial; font-size: 12px; padding: 6px;'
        )
    )
).add_to(m)

colormap.add_to(m)
folium.LayerControl().add_to(m)

# Legend note
legend_html = '''
<div style="position:fixed; bottom:40px; left:40px; z-index:1000;
            background:white; padding:10px 14px; border-radius:6px;
            border:1px solid #ccc; font-size:12px; font-family:Arial; line-height:1.6">
  <b>Bay Area Gentrification Risk</b><br>
  <span style="color:#888">&#9632;</span> Gray = tract outside model<br>
  <span style="color:#ffffff; background:#ff4444; padding:0 4px">&#9632;</span> Red = highest relative risk<br>
  <span style="color:#aaa">&#9632;</span> White = lowest relative risk<br>
  <hr style="margin:4px 0">
  Model: XGBoost + LR + RF ensemble<br>
  Features: 2020→2024 ACS change variables
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

print('Map built. Displaying below...')
m

## 7 — Save & Download Map

In [ ]:
OUTPUT_PATH = '/content/gentrification_risk_map.html'
m.save(OUTPUT_PATH)
print(f'Map saved to {OUTPUT_PATH}')

# Download to your computer
from google.colab import files
files.download(OUTPUT_PATH)

## 8 — Top & Bottom 10 Tracts (for reference)

In [ ]:
scored_tracts = merged[merged['in_model']].copy()
scored_tracts = scored_tracts.sort_values('scaled_risk', ascending=False)

display_cols = ['tract_id', 'city', 'scaled_risk_disp', 'risk_score_disp', 'top_risk_driver']

print('Top 10 highest-risk tracts:')
print(scored_tracts[display_cols].head(10).to_string(index=False))

print('\nTop 10 lowest-risk tracts:')
print(scored_tracts[display_cols].tail(10).to_string(index=False))